In [1]:
!pip install transformers

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from transformers import BertTokenizer, TFBertModel

import numpy as np
import os
import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
os.environ["WANDB_API_KEY"] = "0" ## to silence warning

In [ ]:
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.experimental.TPUStrategy(tpu)
except ValueError:
    strategy = tf.distribute.get_strategy() # for CPU and single GPU
    print('Number of replicas:', strategy.num_replicas_in_sync)

In [ ]:
# hyperparameters
max_length = 140
batch_size = 32
dev_size = 0.3

In [ ]:
# Bert Tokenizer
model_name = "bert-base-multilingual-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

In [ ]:
import pandas as pd

train_df = pd.read_csv("/content/drive/MyDrive/shashank_paper_2/Bhojpuri/merged_data.csv", header=0, names=['label', 'text'])
train_df.head()
#train_df = pd.read_excel('/content/drive/MyDrive/paper_5/Experiment with Hindi Dataset/list_hinglish.xlsx', names = ['CM', 'Hindi', 'label'])

In [ ]:
train_df.tail()

In [ ]:
print(train_df.shape, )

In [ ]:
train, dev = train_test_split(train_df, test_size=dev_size, random_state=42)

In [ ]:
print(dev.shape)

In [ ]:
def bert_encode(data):
    tokens = tokenizer.batch_encode_plus(data, max_length=max_length, padding='max_length', truncation=True)

    return tf.constant(tokens['input_ids'])

In [13]:
train_encoded = bert_encode(train.text)
dev_encoded = bert_encode(dev.text)


train_dataset = (
    tf.data.Dataset
    .from_tensor_slices((train_encoded, train.label))
    .shuffle(100)
    .batch(batch_size)
)

dev_dataset = (
    tf.data.Dataset
    .from_tensor_slices((dev_encoded, dev.label))
    .shuffle(100)
    .batch(batch_size)
)

In [18]:
pip install --upgrade tensorflow transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 100.2 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.17.1
    Uninstalling tensorboard-2.17.1:
      Successfully uninstalled tensorboard-2.17.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.47.1
    Uninstalling transformers-4.47.1:
      Successfully uninstalled transformers-4.47.1
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.17.1
    Uninstalling tensorflow-2.17.1:
      Successfully uninstalled tensorflow-2.17.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depend

In [17]:
import tensorflow as tf
from transformers import BertTokenizer, TFBertModel

# Define constants
model_name = "bert-base-multilingual-uncased"  # Replace with your BERT model name if needed
max_length = 140  # Define the maximum length of input sequences

# Define Bahdanau Attention mechanism
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, features, hidden):
        # features shape == (batch_size, max_length, hidden_size)
        # hidden shape == (batch_size, hidden_size)
        hidden_with_time_axis = tf.expand_dims(hidden, 1)
        score = tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis))
        attention_weights = tf.nn.softmax(self.V(score), axis=1)
        context_vector = attention_weights * features
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

# Define the model with LSTM and Attention
def lstm_with_attention_model():
    # Load the pre-trained BERT model
    bert_encoder = TFBertModel.from_pretrained(model_name)

    # Define input layer for BERT
    input_word_ids = tf.keras.Input(shape=(max_length,), dtype=tf.int32, name="input_ids")

    # Get BERT embeddings
    bert_output = bert_encoder(input_word_ids)
    bert_sequence_output = bert_output.last_hidden_state  # Shape: (batch_size, max_length, hidden_size)

    # Define LSTM layer
    lstm_layer = tf.keras.layers.LSTM(100, return_sequences=True)
    lstm_output = lstm_layer(bert_sequence_output)  # Shape: (batch_size, max_length, 100)

    # Apply attention mechanism
    attention_layer = BahdanauAttention(units=100)
    context_vector, attention_weights = attention_layer(lstm_output, lstm_output[:, -1, :])  # Last hidden state as query

    # Add output layer
    output = tf.keras.layers.Dense(1, activation='sigmoid')(context_vector)

    # Define the model
    model = tf.keras.Model(inputs=input_word_ids, outputs=output)
    return model

# Initialize the distribution strategy if needed
strategy = tf.distribute.get_strategy()

# Build and compile the model
with strategy.scope():
    model = lstm_with_attention_model()
    adam_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5)
    model.compile(loss='binary_crossentropy', optimizer=adam_optimizer, metrics=['accuracy'])

# Print the model summary
model.summary()


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

ValueError: Exception encountered when calling layer 'tf_bert_model_3' (type TFBertModel).

Data of type <class 'keras.src.backend.common.keras_tensor.KerasTensor'> is not allowed only (<class 'tensorflow.python.framework.tensor.Tensor'>, <class 'bool'>, <class 'int'>, <class 'transformers.utils.generic.ModelOutput'>, <class 'tuple'>, <class 'list'>, <class 'dict'>, <class 'numpy.ndarray'>) is accepted for input_ids.

Call arguments received by layer 'tf_bert_model_3' (type TFBertModel):
  • input_ids=<KerasTensor shape=(None, 140), dtype=int32, sparse=False, name=input_ids>
  • attention_mask=None
  • token_type_ids=None
  • position_ids=None
  • head_mask=None
  • inputs_embeds=None
  • encoder_hidden_states=None
  • encoder_attention_mask=None
  • past_key_values=None
  • use_cache=None
  • output_attentions=None
  • output_hidden_states=None
  • return_dict=None
  • training=False

In [ ]:
train_dataset

In [ ]:
#from keras.utils.vis_utils import plot_model
with strategy.scope():
    model = lstm_with_attention_model()
    adam_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5)
    model.compile(loss='binary_crossentropy',optimizer=adam_optimizer,metrics=['accuracy'])

    model.summary()
    #plot_model(model=model, show_shapes=True)


In [ ]:
history = model.fit(
    train_dataset,
    batch_size=batch_size,
    epochs=3,
    validation_data=dev_dataset,
    verbose=2)
    #callbacks=[tf.keras.callbacks.EarlyStopping(
    #            patience=6,
    #            min_delta=0.05,
    #            baseline=0.7,
    #            mode='min',
    #            monitor='val_accuracy',
    #            restore_best_weights=True,
    #            verbose=1)
    #          ])

In [ ]:
import matplotlib.pyplot as plt

def plot_graphs(history, string):
  plt.plot(history.history[string])
  plt.plot(history.history['val_'+string])
  plt.xlabel("Epochs")
  plt.ylabel(string)
  plt.legend([string, 'val_'+string])
  plt.show()

plot_graphs(history, "accuracy")
plot_graphs(history, "loss")

In [ ]:
test, val = train_test_split(dev, test_size=dev_size, random_state=42)

In [ ]:
dev.shape

In [ ]:
test.head()

In [ ]:
test_encoded = bert_encode(dev.text)

test_dataset = (
    tf.data.Dataset
    .from_tensor_slices(test_encoded)
    .batch(batch_size)
)

predicted_tweets = model.predict(test_dataset, batch_size=batch_size)
predicted_tweets_binary = tf.cast(tf.round(predicted_tweets), tf.int32).numpy().flatten()
print(predicted_tweets_binary)
#my_submission = pd.DataFrame({'id': dev.id, 'label': predicted_tweets_binary})
#my_submission.to_csv('my_submission.csv', index=False)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn import metrics

In [ ]:
print("Bert+CNN")
print("Accuracy score =", accuracy_score(dev['label'], predicted_tweets_binary))
print(metrics.classification_report(dev['label'], predicted_tweets_binary))


Bert+LSTM
Accuracy score = 0.8930451648656137
              precision    recall  f1-score   support

           0       0.93      0.89      0.91      2171
           1       0.84      0.90      0.87      1438

    accuracy                           0.89      3609
   macro avg       0.89      0.89      0.89      3609
weighted avg       0.90      0.89      0.89      3609



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    RocCurveDisplay,
    precision_recall_curve,
    PrecisionRecallDisplay,
)

# Sample true labels and predicted probabilities
# Replace these with your actual data
#y_true = np.array([0, 1, 1, 0, 1, 0, 1, 1, 0, 0])
#y_pred_proba = np.array([0.1, 0.8, 0.9, 0.3, 0.7, 0.2, 0.85, 0.9, 0.4, 0.5])
#y_pred = (y_pred_proba >= 0.5).astype(int)

# Plot Confusion Matrix
cm = confusion_matrix(dev['label'], predicted_tweets_binary)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()
#plt.savefig('/content/drive/MyDrive/shashank_paper_2/Hindi//confusion_h_bert-cnn.pdf', format="pdf")

# Plot ROC Curve
fpr, tpr, _ = roc_curve(dev['label'], predicted_tweets_binary)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"ROC Curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.tight_layout()
plt.legend(loc="lower right")
plt.show()
#plt.savefig('/content/drive/MyDrive/shashank_paper_2/Hindi//roc_h_bert-cnn.pdf', format="pdf")


NameError: name 'dev' is not defined

In [ ]:
non_dp = train_df[train_df['level'] == 0]  # DataFrame for level == 0
dp = train_df[train_df['level'] == 1]

NameError: name 'train_df' is not defined

In [ ]:
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt
stopwords = set(STOPWORDS)

def show_wordcloud(data, title = None):
    wordcloud = WordCloud(
        background_color='white',
        stopwords=stopwords,
        max_words=200,
        max_font_size=40,
        scale=3,
        random_state=1 # chosen at random by flipping a coin; it was heads
    ).generate(str(data))

    fig = plt.figure(1, figsize=(12, 12))
    plt.axis('on')
    if title:
        fig.suptitle(title, fontsize=20)
        fig.subplots_adjust(top=2.3)

    plt.imshow(wordcloud)
    plt.show()

show_wordcloud(r_sarcastic['text'])
#wordcloud(sarcastic_train['text']).to_file('Non-sarcastic.png')
show_wordcloud(r_n_sarcastic['text'])
#wordcloud(nsarcastic_train['text']).to_file('sarcastic.png')